In [2]:
import pandas as pd
from pathlib import Path

RAW = Path("/data/raw")
OUT = Path("/data/interim")
OUT.mkdir(parents=True, exist_ok=True)

ALL_MONTHS = pd.date_range("2019-01-01", "2024-12-01", freq="MS")


def main():
    print("Loading FEMA Disaster Declarations...")
    df = pd.read_csv(RAW / "FEMA_disaster_declarations_summaries.csv")

    # ── Filter to Puerto Rico, 2019–2024 ─────────────────────────────────────
    pr = df[df["state"] == "PR"].copy()
    pr["incidentBeginDate"] = pd.to_datetime(pr["incidentBeginDate"], utc=True)
    pr["declarationDate"]   = pd.to_datetime(pr["declarationDate"],   utc=True)
    pr["year"] = pr["incidentBeginDate"].dt.year
    pr = pr[pr["year"].between(2019, 2024)].copy()

    # ── Deduplicate to unique disaster events (raw file is county-level) ──────
    events = pr.drop_duplicates(subset=["disasterNumber"]).copy()
    events["event_month"] = events["incidentBeginDate"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()

    print(f"  Unique disaster events 2019-2024: {len(events)}")
    print(f"  Events by year:\n{events.groupby(events['incidentBeginDate'].dt.year).size().to_string()}")

    # ── Aggregate to month ────────────────────────────────────────────────────
    monthly_events = events.groupby("event_month").agg(
        disaster_count  = ("disasterNumber",    "nunique"),
        pa_declared     = ("paProgramDeclared", "max"),
        ia_declared     = ("iaProgramDeclared", "max"),
        major_dr_declared = ("declarationType", lambda x: int("DR" in x.values)),
        incident_types  = ("incidentType",      lambda x: ",".join(sorted(x.unique()))),
    ).reset_index().rename(columns={"event_month": "month"})

    # ── Build full 2019–2024 monthly grid and merge ───────────────────────────
    grid = pd.DataFrame({"month": ALL_MONTHS})
    grid["iso3"] = "PRI"
    grid["year"] = grid["month"].dt.year

    result = grid.merge(monthly_events, on="month", how="left")
    result["disaster_count"]    = result["disaster_count"].fillna(0).astype(int)
    result["pa_declared"]       = result["pa_declared"].fillna(0).astype(int)
    result["ia_declared"]       = result["ia_declared"].fillna(0).astype(int)
    result["major_dr_declared"] = result["major_dr_declared"].fillna(0).astype(int)
    result["incident_types"]    = result["incident_types"].fillna("")
    result["fema_event_flag"]   = result["disaster_count"].apply(
        lambda x: "fema_disaster" if x > 0 else "fema_no_event"
    )

    result = result.sort_values("month").reset_index(drop=True)

    # ── QA ────────────────────────────────────────────────────────────────────
    print(f"\n=== QA SUMMARY ===")
    print(f"Total rows  : {len(result)} (expected 72)")
    print(f"Nulls       : {result.isnull().sum().sum()}")
    print(f"Event months: {result['fema_event_flag'].value_counts().to_string()}")
    print(f"\nDisaster months:")
    active = result[result["disaster_count"] > 0]
    print(active[["month","disaster_count","incident_types","pa_declared",
                   "major_dr_declared"]].to_string(index=False))

    # ── Write ─────────────────────────────────────────────────────────────────
    out_cols = ["iso3","year","month","disaster_count","pa_declared",
                "ia_declared","major_dr_declared","incident_types","fema_event_flag"]
    result[out_cols].to_csv(OUT / "fema_country_month.csv", index=False)
    print(f"\n  Wrote fema_country_month.csv ({len(result)} rows)")


if __name__ == "__main__":
    main()

Loading FEMA Disaster Declarations...
  Unique disaster events 2019-2024: 15
  Events by year:
incidentBeginDate
2019    3
2020    6
2022    3
2024    3

=== QA SUMMARY ===
Total rows  : 72 (expected 72)
Nulls       : 0
Event months: fema_event_flag
fema_no_event    62
fema_disaster    10

Disaster months:
     month  disaster_count              incident_types  pa_declared  major_dr_declared
2019-08-01               1                Severe Storm            1                  0
2019-12-01               2                  Earthquake            1                  1
2020-01-01               2                  Biological            1                  1
2020-07-01               2                   Hurricane            1                  1
2020-08-01               1                   Hurricane            1                  0
2020-09-01               1                       Flood            0                  1
2022-02-01               1                       Flood            0                